# rPPG Training Notebook

This notebook drives the full **train → validate → test** pipeline for all rPPG models  
by calling `main.py` with the appropriate YAML config files located in `configs/train_configs/`.

**Workflow:**
1. Set paths and pick a training config
2. (Optional) Verify GPU and environment
3. Run training via `main.py --config_file <yaml>`
4. (Optional) Run only-test with a pretrained checkpoint
5. Aggregate and display results

---

**Model → Trainer mapping** (all under `neural_methods/trainer/`):

| Config MODEL.NAME | Trainer file            | Group |
|-------------------|-------------------------|-------|
| `Tscan`           | TscanTrainer.py         | A     |
| `DeepPhys`        | DeepPhysTrainer.py      | A     |
| `EfficientPhys`   | EfficientPhysTrainer.py | B     |
| `Physnet`         | PhysnetTrainer.py       | C     |
| `PhysFormer`      | PhysFormerTrainer.py    | D     |
| `PhysMamba`       | PhysMambaTrainer.py     | E     |
| `iBVPNet`         | iBVPNetTrainer.py       | F     |
| `FactorizePhys`   | FactorizePhysTrainer.py | F     |
| `RhythmFormer`    | RhythmFormerTrainer.py  | G     |
| `BigSmall`        | BigSmallTrainer.py      | BigSmall |

In [ ]:
import os
import sys
import subprocess
import json
import glob
import pandas as pd

# ─── REPO ROOT ───────────────────────────────────────────────────────────────
# Update this to the absolute path of your rPPG folder
REPO_ROOT = "/home/iec/MinhHieu/rPPG"

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"Python    : {sys.executable}")

In [ ]:
# ─── GPU CHECK ────────────────────────────────────────────────────────────────
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}  ({props.total_memory // 1024**2} MB)")

---
## Step 1 — Select a training config

Choose one config from `configs/train_configs/` and set `TRAIN_CONFIG` below.  
Common configs by group:

| Group | Example config |
|-------|----------------|
| A (DeepPhys/TSCAN) | `PURE_PURE_UBFC-rPPG_TSCAN_BASIC.yaml` |
| A (DeepPhys/TSCAN) | `UBFC-rPPG_UBFC-rPPG_PURE_DEEPPHYS_BASIC.yaml` |
| B (EfficientPhys)  | `PURE_PURE_UBFC-rPPG_EFFICIENTPHYS.yaml` |
| C (PhysNet)        | `PURE_PURE_UBFC-rPPG_PHYSNET_BASIC.yaml` |
| D (PhysFormer)     | `PURE_PURE_UBFC-rPPG_PHYSFORMER_BASIC.yaml` |
| E (PhysMamba)      | `PURE_PURE_UBFC-rPPG_PHYSMAMBA_BASIC.yaml` |
| F (iBVPNet)        | `PURE_PURE_iBVP_iBVPNet_BASIC.yaml` |
| F (FactorizePhys)  | `PURE_iBVP_FactorizePhys_FSAM_Res.yaml` |
| G (RhythmFormer)   | `PURE_PURE_UBFC-rPPG_RHYTHMFORMER_BASIC.yaml` |

In [ ]:
# ─── CONFIG SELECTION ────────────────────────────────────────────────────────
TRAIN_CONFIG = "configs/train_configs/PURE_PURE_UBFC-rPPG_TSCAN_BASIC.yaml"

config_path = os.path.join(REPO_ROOT, TRAIN_CONFIG)
assert os.path.exists(config_path), f"Config not found: {config_path}"

print(f"Selected config : {TRAIN_CONFIG}")
print(f"Full path       : {config_path}")

# Preview the config file
with open(config_path) as f:
    print("\n--- Config preview ---")
    print(f.read())

In [ ]:
# ─── LIST ALL AVAILABLE TRAIN CONFIGS ────────────────────────────────────────
all_configs = sorted(glob.glob(os.path.join(REPO_ROOT, "configs/train_configs/*.yaml")))
print(f"Found {len(all_configs)} training configs:\n")
for c in all_configs:
    print(" ", os.path.basename(c))

---
## Step 2 — Run Training (`train_and_test` mode)

> **Before running:** open the selected YAML and update `DATA_PATH` and `CACHED_PATH`  
> to point to your local dataset and preprocessed-data directories.
>
> Set `DO_PREPROCESS: True` on the **first** run so raw data gets preprocessed and cached.  
> Switch it to `False` afterwards to reuse the cached files and speed up subsequent runs.

In [ ]:
# ─── TRAIN + TEST ─────────────────────────────────────────────────────────────
cmd = [
    sys.executable,
    os.path.join(REPO_ROOT, "main.py"),
    "--config_file", config_path,
]

print("Running command:")
print(" ".join(cmd))
print("=" * 70)

result = subprocess.run(
    cmd,
    cwd=REPO_ROOT,
    capture_output=False,   # stream output directly to notebook
    text=True,
)

print("=" * 70)
print(f"Return code: {result.returncode}")
if result.returncode != 0:
    print("[ERROR] Training failed. Check output above for details.")

---
## Step 3 — Run multiple configs in batch (optional)

Loop over a list of configs to train several models back-to-back.  
Results are logged per-run.

In [ ]:
# ─── BATCH TRAINING ───────────────────────────────────────────────────────────
# Edit this list to train multiple models sequentially
BATCH_CONFIGS = [
    "configs/train_configs/PURE_PURE_UBFC-rPPG_TSCAN_BASIC.yaml",
    "configs/train_configs/PURE_PURE_UBFC-rPPG_DEEPPHYS_BASIC.yaml",
    # "configs/train_configs/PURE_PURE_UBFC-rPPG_EFFICIENTPHYS.yaml",
    # "configs/train_configs/PURE_PURE_UBFC-rPPG_PHYSNET_BASIC.yaml",
    # "configs/train_configs/PURE_PURE_UBFC-rPPG_PHYSFORMER_BASIC.yaml",
    # "configs/train_configs/PURE_PURE_UBFC-rPPG_RHYTHMFORMER_BASIC.yaml",
]

batch_results = []

for cfg_rel in BATCH_CONFIGS:
    cfg_abs = os.path.join(REPO_ROOT, cfg_rel)
    print(f"\n{'='*70}")
    print(f"Training: {os.path.basename(cfg_rel)}")
    print(f"{'='*70}")

    if not os.path.exists(cfg_abs):
        print(f"  [SKIP] Config not found: {cfg_abs}")
        batch_results.append({"config": cfg_rel, "status": "not_found", "returncode": -1})
        continue

    res = subprocess.run(
        [sys.executable, os.path.join(REPO_ROOT, "main.py"), "--config_file", cfg_abs],
        cwd=REPO_ROOT,
        capture_output=False,
        text=True,
    )
    status = "success" if res.returncode == 0 else "failed"
    batch_results.append({"config": os.path.basename(cfg_rel), "status": status, "returncode": res.returncode})
    print(f"  → {status} (return code {res.returncode})")

print("\n--- Batch summary ---")
pd.DataFrame(batch_results)

---
## Step 4 — Only-test with a pretrained checkpoint

Use an `infer_config` YAML from `configs/infer_configs/` and provide the `.pth` weight path.

In [ ]:
# ─── ONLY TEST ────────────────────────────────────────────────────────────────
INFER_CONFIG  = "configs/infer_configs/PURE_UBFC-rPPG_TSCAN_BASIC.yaml"
WEIGHTS_PATH  = "final_model_release/PURE_TSCAN.pth"           # relative to REPO_ROOT

infer_cfg_abs  = os.path.join(REPO_ROOT, INFER_CONFIG)
weights_abs    = os.path.join(REPO_ROOT, WEIGHTS_PATH)

assert os.path.exists(infer_cfg_abs), f"Infer config not found: {infer_cfg_abs}"
assert os.path.exists(weights_abs),   f"Weights not found: {weights_abs}"

cmd_test = [
    sys.executable,
    os.path.join(REPO_ROOT, "main.py"),
    "--config_file", infer_cfg_abs,
]

print("Running only-test:")
print(" ".join(cmd_test))

result_test = subprocess.run(cmd_test, cwd=REPO_ROOT, capture_output=False, text=True)
print(f"Return code: {result_test.returncode}")

---
## Step 5 — Aggregate saved test outputs

After training/testing, results are saved under `runs/exp/<exp_name>/saved_test_outputs/`.  
Use this cell to read and display them.

In [ ]:
# ─── READ SAVED OUTPUTS ───────────────────────────────────────────────────────
import pickle
import numpy as np

OUTPUT_DIR = os.path.join(REPO_ROOT, "runs", "exp")

pickle_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "**", "*.pickle"), recursive=True))
print(f"Found {len(pickle_files)} saved output file(s):")
for p in pickle_files:
    print(" ", os.path.relpath(p, REPO_ROOT))

In [ ]:
# ─── DISPLAY RESULTS FROM A SPECIFIC OUTPUT FILE ─────────────────────────────
# Update this path to one of the pickle files found above
RESULT_FILE = pickle_files[0] if pickle_files else None

if RESULT_FILE is None:
    print("No result files found. Run training first.")
else:
    with open(RESULT_FILE, "rb") as f:
        data = pickle.load(f)

    predictions = data["predictions"]  # dict: subj_id -> {chunk_id: np.array}
    labels      = data["labels"]       # same structure
    label_type  = data.get("label_type", "unknown")
    fs          = data.get("fs", 30)

    print(f"File      : {os.path.relpath(RESULT_FILE, REPO_ROOT)}")
    print(f"Subjects  : {len(predictions)}")
    print(f"Label type: {label_type}")
    print(f"Sampling  : {fs} Hz")
    print(f"Subject IDs: {sorted(predictions.keys())}")

In [ ]:
# ─── COMPUTE HR METRICS FROM SAVED OUTPUTS ────────────────────────────────────
from scipy import signal as scipy_signal
from scipy.signal import periodogram

def detrend(sig, lam=100):
    T = len(sig)
    H = np.eye(T)
    ones = np.ones(T)
    D = np.diag(ones[:-2], -2) - 2*np.diag(ones[:-1], -1) + np.diag(ones)
    D = D[2:, :]
    return (H - np.linalg.inv(H + lam**2 * D.T @ D)) @ sig

def bandpass(sig, fs, lo=0.6, hi=3.3, order=1):
    b, a = scipy_signal.butter(order, [lo/(fs/2), hi/(fs/2)], btype="bandpass")
    return scipy_signal.filtfilt(b, a, sig.astype(np.float64))

def fft_hr(sig, fs, lo=0.6, hi=3.3):
    N = 1
    while N < len(sig): N *= 2
    freqs, pxx = periodogram(sig, fs=fs, nfft=N, detrend=False)
    mask = (freqs >= lo) & (freqs <= hi)
    return float(freqs[mask][np.argmax(pxx[mask])]) * 60.0 if mask.any() else 0.0

def reform(chunk_dict):
    return np.concatenate([chunk_dict[k] for k in sorted(chunk_dict.keys())])

if RESULT_FILE:
    diff_flag = (label_type == "DiffNormalized")
    rows = []

    for subj in sorted(predictions.keys()):
        pred  = reform(predictions[subj]).astype(np.float64)
        label = reform(labels[subj]).astype(np.float64)

        if diff_flag:
            pred  = detrend(np.cumsum(pred))
            label = detrend(np.cumsum(label))
        else:
            pred  = detrend(pred)
            label = detrend(label)

        pred_bp  = bandpass(pred,  fs)
        label_bp = bandpass(label, fs)

        hr_pred  = fft_hr(pred_bp,  fs)
        hr_label = fft_hr(label_bp, fs)

        rows.append({"subject": subj, "HR_pred": hr_pred, "HR_label": hr_label,
                     "HR_error": hr_pred - hr_label})

    df_results = pd.DataFrame(rows)

    mae  = df_results["HR_error"].abs().mean()
    rmse = (df_results["HR_error"]**2).mean()**0.5
    r    = df_results[["HR_pred","HR_label"]].corr().iloc[0,1]

    print(f"\n--- Results: {os.path.basename(RESULT_FILE)} ---")
    print(f"MAE     : {mae:.2f} bpm")
    print(f"RMSE    : {rmse:.2f} bpm")
    print(f"Pearson : {r:.4f}")
    print()
    print(df_results.to_string(index=False))

---
## Step 6 — Compare all trained models (aggregate leaderboard)

Scans all `*.pickle` files under `runs/exp/` and builds a comparison table.

In [ ]:
# ─── LEADERBOARD ACROSS ALL SAVED OUTPUTS ─────────────────────────────────────
all_rows = []

for pkl_path in pickle_files:
    with open(pkl_path, "rb") as f:
        d = pickle.load(f)

    preds_all  = d["predictions"]
    labels_all = d["labels"]
    lt  = d.get("label_type", "DiffNormalized")
    fps = d.get("fs", 30)
    diff = (lt == "DiffNormalized")

    hr_ps, hr_ls = [], []
    for subj in preds_all:
        p = reform(preds_all[subj]).astype(np.float64)
        l = reform(labels_all[subj]).astype(np.float64)
        if diff:
            p = detrend(np.cumsum(p)); l = detrend(np.cumsum(l))
        else:
            p = detrend(p); l = detrend(l)
        hr_ps.append(fft_hr(bandpass(p, fps), fps))
        hr_ls.append(fft_hr(bandpass(l, fps), fps))

    hr_ps  = np.array(hr_ps)
    hr_ls  = np.array(hr_ls)
    errs   = hr_ps - hr_ls
    mae    = float(np.abs(errs).mean())
    rmse   = float((errs**2).mean()**0.5)
    pearson = float(np.corrcoef(hr_ps, hr_ls)[0, 1]) if len(hr_ps) >= 2 else float("nan")

    model_name = os.path.basename(pkl_path).replace("_outputs.pickle", "")
    all_rows.append({"model": model_name, "n": len(hr_ps),
                     "MAE (bpm)": round(mae, 2), "RMSE (bpm)": round(rmse, 2),
                     "Pearson": round(pearson, 4)})

if all_rows:
    df_board = pd.DataFrame(all_rows).sort_values("MAE (bpm)").reset_index(drop=True)
    df_board.insert(0, "rank", df_board.index + 1)
    print(df_board.to_string(index=False))
    df_board.to_csv(os.path.join(OUTPUT_DIR, "leaderboard.csv"), index=False)
    print(f"\nSaved leaderboard to: {os.path.join(OUTPUT_DIR, 'leaderboard.csv')}")
else:
    print("No result files found. Run training or testing first.")